In [ ]:
import pandas as pd
import json
import glob
import datetime
import PySimpleGUI as sg
import ast
import re
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pygsheets
import textwrap
import subprocess
import sys
import os
import pyglet,tkinter
# import OpenGL
# from OpenGL import GLU
pyglet.font.add_file('/etc/fonts/fonts/CENTAUR.TTF')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

## INIT

In [ ]:
## RUN THESE AT STARTUP
stats={}
statsYrMo = {}
statsYrMoDy = {}
statsYrMoName = {}
statsNameYrMo = {}
statsMetaDataYrMoDy={}
statsYrMoValues = []


if sys.platform == "linux":
    path="logs"
else:
    path= "\\\\wsl.localhost\\Ubuntu\\home\\joe\\work\\Logs\\"



## Update Log Files

In [ ]:
def updateLogs(how=1):
    #  how    1 = update just this month (log.json.nn
    #         2 = update Last Months file (log_backup_year_mo.json
    #         3 = update all this years log files 
    pw=os.environ["OIT_PW"]
    #  compute previous month
    x = datetime.datetime.today() - datetime.timedelta(days=30)
    monthM = x.month  
    yearM = x.year
    
    year = datetime.datetime.today().year
    
    string = f'''sshpass -p {pw} sftp -o HostKeyAlgorithms=+ssh-rsa -o PubkeyAcceptedAlgorithms=+ssh-rsa  giddensm@165.127.62.8 << !'''

    if how == 1:  #  update this month
        string+= "\nmget /usr/local/cim/bic_etl/general/logs/log.json.[0-9]* logs"
##  remove all of this months logs first to account for month changes
        for x in os.listdir(path):
            if  re.findall("^log.json\.\d+",x):
                 file=f"{path}/{x}"
                 print("Removing ",file)
                 os.remove(file)
        
    elif how == 2: 
        string+=f"\nmget /usr/local/cim/bic_etl/general/logs/log_backup_{yearM}_{monthM}*.json logs"
    elif how == 3:
        string+=f"\nmget /usr/local/cim/bic_etl/general/logs/log_backup_{year}*.json logs"
    
    string+="\n!'''"

    print(string)
    result=subprocess.run([string], shell=True, stdout=subprocess.PIPE)
    x = str(result.stdout)
    nfiles=0
    files=x.split("\\n")
    filesOut=[]
    for line in files:
        print(line)
        if "fetch" in line.lower():
            nfiles+=1
            filesOut.append(line)
    readLogs(path)        
            
    return nfiles,filesOut
    

## Get Old Log Files

In [ ]:
## Processs the log files and extract out the errors



def readLogs(path):

    global stats,statsYrMo,statsYrMoDy,statsYrMoValues,statsYrMoName,statsNameYrMo,statsMetaDataYrMoDy 
    stats={}
    statsYrMo = {}
    statsYrMoDy = {}
    statsYrMoName = {}
    statsNameYrMo = {}
    statsMetaDataYrMoDy={}
    statsYrMoValues = []
    
    nfail=0
    nfile=0
#  get files to process
    files2Process = []
    for x in os.listdir(path):
        if re.findall("\d{4}_\d+.json$",x) or re.findall("^log.json\.\d+",x):
             files2Process.append(x)
                
    fout = open("data/error-log.json","w")
    bad = []
    #statsYrMoDyDf = pd.DataFrame(columns=["Year","Month","Jan","Feb","Mar","Apr","May","Jun","Aug","Sep","Oct","Nov","Dec"])

    for yr in range(2019,2024):
        statsYrMoDy[yr] = {}
        statsMetaDataYrMoDy[yr] = {}

        for mo in range(1,13):
            statsYrMoDy[yr][mo] = {}
            statsMetaDataYrMoDy[yr][mo] = {}        
            for dy in range(1,32):
                statsYrMoDy[yr][mo][dy]=0
                statsMetaDataYrMoDy[yr][mo][dy]={}


    for file in files2Process:
        nfile+=1
        print(file)
        with open(f"{path}/{file}") as jsf:

            try:
                log=[]
                nline=0
                for line in jsf:
                    try: 
           #         log.append(json.loads(line))
              #         fout.write(line)
                       log.append(ast.literal_eval(line))
                    except:
                       bad.append(line)
                    nline+=1

                for line in log:
                    # if (nfile%5 ==0):
                    #    print(line)
                    if line["msg"].lower().find("error") >= 0:
                       

         #               date_object = datetime.strptime(line['time'], '%Y-%m-%dT%h:%M:%s:%fZ').date()
        #                date_object = datetime.strptime(line['time'], '%Y-%m-%d').date()
                        yr = int( line['time'][0:4])
                        mo = int(line['time'][5:7])
                        dy = int(line['time'][8:10])

                        name = line['name']
                        if name == "Metadata Updater":
                             spl = line['msg'].split(" ")
                             ds=spl[3]
                             if ds in statsMetaDataYrMoDy[yr][mo][dy]:
                                statsMetaDataYrMoDy[yr][mo][dy][ds]+= 1
                             else:
                                statsMetaDataYrMoDy[yr][mo][dy][ds]= 1

                        fout.write(f"{str(line)}\n")


                        if yr in statsYrMo and mo in statsYrMo[yr]:
                             statsYrMo[yr][mo]+=1
                        elif yr not in statsYrMo:
                            statsYrMo[yr] = {}
                            statsYrMo[yr][mo]=1
                        elif mo not in statsYrMo[yr]:
                            statsYrMo[yr][mo]=1

                        if yr not in statsYrMoDy:
                                statsYrMoDy[yr] = {}
                        elif mo not in statsYrMoDy[yr]:
                                statsYrMoDy[yr][mo] = {}
                        elif dy not in statsYrMoDy[yr][mo]:
                                statsYrMoDy[yr][mo][dy] = 1
                        else:
                                statsYrMoDy[yr][mo][dy]+= 1


                        if yr in statsYrMoName and mo in statsYrMoName[yr]:
                            if name in statsYrMoName[yr][mo]:
                                statsYrMoName[yr][mo][name]+=1
                            else:
                                statsYrMoName[yr][mo][name]=1
                        elif yr not in statsYrMoName:
                            statsYrMoName[yr] = {}
                            statsYrMoName[yr][mo] = {}
                            statsYrMoName[yr][mo][name]=1
                        elif mo not in statsYrMoName[yr]:
                            statsYrMoName[yr][mo] = {}
                            statsYrMoName[yr][mo][name]=1
                        elif name not in statsYrMoName[yr][mo]:
                            statsYrMoName[yr][mo][name]=1

                        if name in statsNameYrMo and yr in statsNameYrMo[name]:
                            if mo in statsNameYrMo[name][yr]:
                                statsNameYrMo[name][yr][mo]["count"]+=1
                                statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                            else:
                                statsNameYrMo[name][yr][mo] = {}
                                statsNameYrMo[name][yr][mo]["count"]=1
                                statsNameYrMo[name][yr][mo]["errors"]= []
                                statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])

                        elif name not in statsNameYrMo:
                            statsNameYrMo[name] = {}
                            statsNameYrMo[name][yr] = {}
                            statsNameYrMo[name][yr][mo] = {}
                            statsNameYrMo[name][yr][mo]["count"]=1
                            statsNameYrMo[name][yr][mo]["errors"]= []
                            statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                        elif yr not in statsNameYrMo[name]:
                            statsNameYrMo[name][yr] = {}
                            statsNameYrMo[name][yr][mo] = {}
                            statsNameYrMo[name][yr][mo]["count"]=1
                            statsNameYrMo[name][yr][mo]["errors"]= []
                            statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])
                        elif mo not in statsNameYrMo[name][yr]:
                            statsNameYrMo[name][yr][mo] = {}
                            statsNameYrMo[name][yr][mo]["count"]=1
                            statsNameYrMo[name][yr][mo]["errors"]= []
                            statsNameYrMo[name][yr][mo]["errors"].append(line['msg'])



                        if line['name'] in stats:
                            stats[line['name']]+=1
                        else:
                            stats[line['name']]=1
            except Exception as e: 
                print("FAIL ",nfile,nline,file,e)
                nfail+=1

                #            log.append(json.loads(line))

    for yr in sorted( statsYrMo.keys()):
    #       for mo in sorted(statsYrMo[yr].keys()):
            for mo in range(1,13):
                if mo not in statsYrMo[yr]:
                    statsYrMo[yr][mo] = 0 

    
    for yr in sorted(statsYrMo.keys()):
         vals=[]
         vals.append(yr)
         for k,val in sorted(statsYrMo[yr].items()):
            vals.append(val)
         statsYrMoValues.append(vals)
        
 #   print(statsYrMoValues)
    #      
    for yr in statsMetaDataYrMoDy.keys():
        for mo in statsMetaDataYrMoDy[yr].keys():
            for dy in statsMetaDataYrMoDy[yr][mo].keys():
                statsMetaDataYrMoDy[yr][mo][dy]["Count"]=0
                for ds in statsMetaDataYrMoDy[yr][mo][dy]:
                    statsMetaDataYrMoDy[yr][mo][dy]["Count"]+=1
    
    for yr in sorted(statsYrMo.keys()):
       vals=[]
       vals.append(yr)
       for k,val in sorted(statsYrMo[yr].items()):
        vals.append(val)
       statsYrMoValues.append(vals)          
    
    print("-----------------------")
    print(f"Total Files Processed {nfile}")
    print(f"# Files that Failed Processing {nfail}")
    
def getRecentErrors():
    global ids
    files = []
    print(path)
    ids={}
    for x in os.listdir(path):
            if re.findall("^log.json\.\d+",x):
                 # files2Process.append(x)
                 files.append(x)

    x = datetime.datetime.today() - datetime.timedelta(days=30)
    monthM = x.month  
    yearM = x.year
    ff=f"log_backup_{yearM}_{monthM}.json"
    files.append(ff)
    errorsByDate = {}
    errorsByName = {}
    print(files)
    bad = []
    for file in files:
        print(file)
        with open(f"{path}/{file}") as jsf:

            try:
                log=[]
                nline=0
                for line in jsf:
                    try: 
                         xl=line.lower()
                         
                         if "error" in xl and "metadata" not in xl:
                    #     if "error" in xl :
                             log.append(ast.literal_eval(line))
                    except:
                       bad.append(line)
                       print("BAD ",line)
                    nline+=1

                for line in log:
                  #  print(line['time'],line['name'])
                    tim = line['time']
                   
                    spl = tim.split("T")
                    tim = spl[0]
                    name = line['name'].strip()
                    msg= line['msg'].strip()
                    uid = f"{line['time']}{msg[1:20]}"
                    if uid in ids:
                        ids[uid]+=1
                    else:
                        ids[uid]=1

                    stitle=""
                    xm = msg.find("Full ETL failure for ")
                    if xm > -1:
                       stitle = msg[xm+20:].split(":")[0]
                       stitle=stitle.replace(" at load","")

                    xm = msg.find("Error Loading ")
                    if xm > -1:
                       stitle = msg[xm+14:]

                    xm = msg.find("Error Extracting ")
                    if xm > -1:
                       stitle=msg[xm+16:]

                    xm = msg.find("Error Transforming ")
                    if xm > -1:
                       stitle=msg[xm+18:]

                #   
                    titl=""
                    s4x4=""
                    w4x4=""
                    try:
                        s4x4s = re.findall("[\w]{3,4}-[\w]{3,4}",line['msg'])
                       

                        if len(s4x4s) > 0:
                          for s4x4 in s4x4s:
                             if s4x4 in xrefsBy4x4:
                                titl = xrefsBy4x4[s4x4]
                                w4x4=s4x4
                                

                    except:
                        titl=""


                    if tim in errorsByDate:
                        errorsByDate[tim]['name'].append(name)
                        errorsByDate[tim]['msg'].append(msg)
                        errorsByDate[tim]['line'].append(line)
                        errorsByDate[tim]['file'].append(file)
                        errorsByDate[tim]['title'].append(titl)
                        errorsByDate[tim]['stitle'].append(stitle)
                        errorsByDate[tim]['4x4'].append(w4x4)
                        errorsByDate[tim]['uid'].append(uid)
                        

                    else:
                        errorsByDate[tim] = {}
                        errorsByDate[tim]['name'] = []
                        errorsByDate[tim]['msg'] = []
                        errorsByDate[tim]['line'] = []
                        errorsByDate[tim]['file'] = []
                        errorsByDate[tim]['title'] = []
                        errorsByDate[tim]['stitle'] = []
                        errorsByDate[tim]['4x4'] = []
                        errorsByDate[tim]['uid'] = []
                        



                        errorsByDate[tim]['name'].append(name)
                        errorsByDate[tim]['msg'].append(msg)
                        errorsByDate[tim]['line'].append(line)
                        errorsByDate[tim]['file'].append(file)
                        errorsByDate[tim]['title'].append(titl)
                        errorsByDate[tim]['stitle'].append(stitle)
                        errorsByDate[tim]['4x4'].append(w4x4)
                        errorsByDate[tim]['uid'].append(uid)
                        


                    if name in errorsByName:
                            errorsByName[name]['time'].append(tim)
                            errorsByName[name]['msg'].append(msg)
                            errorsByName[name]['line'].append(line)
                            errorsByName[name]['file'].append(file)
                            errorsByName[name]['title'].append(titl)
                    else:
                        errorsByName[name] = {}
                        errorsByName[name]['time'] = []
                        errorsByName[name]['msg'] = []
                        errorsByName[name]['line'] = []
                        errorsByName[name]['file'] = []
                        errorsByName[name]['title'] = []

                        errorsByName[name]['time'].append(tim)
                        errorsByName[name]['msg'].append(msg)
                        errorsByName[name]['line'].append(line)
                        errorsByName[name]['file'].append(file)
                        errorsByName[name]['title'].append(titl)

            except Exception as err:
                print(err)
                print(line)
    return errorsByDate,errorsByName
    

    
readLogs(path)

## GUI Functions


In [ ]:
def getXrefs():
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('../client_secret.json',
     scope)
    client = gspread.authorize(creds)

    gc = gspread.service_account("../client_secret.json")
    for gg in gc.list_spreadsheet_files():
         print("GGGGG ",gg)
    # https://docs.google.com/spreadsheets/d/1WTaOglzbSsYiHhAGguGxHQXmAGmOhfFHkGkMLowxAOA/edit?usp=sharing
    sheet = client.open('BIC Data Inventory and Metadata').worksheet(
        'Maintenance_Framework')
    repo_sheet = client.open('BIC Data Inventory and Metadata').worksheet(
        'MetadataRepository')
    fields_sheet = client.open('BIC Data Inventory and Metadata').worksheet(
        'Field Descriptions')
    
    
#     sheet = client.open('https://docs.google.com/spreadsheets/d/1Xc7oYpdCLwHmUCaokpfXkF4bzkwRW0xUbvBZwruF0ZI/').worksheet(
#         'Maintenance_Framework')
#     repo_sheet = client.open('https://docs.google.com/spreadsheets/d/1Xc7oYpdCLwHmUCaokpfXkF4bzkwRW0xUbvBZwruF0ZI/').worksheet(
#         'MetadataRepository')
    
#     fields_sheet = client.open('https://docs.google.com/spreadsheets/d/1Xc7oYpdCLwHmUCaokpfXkF4bzkwRW0xUbvBZwruF0ZI/').worksheet(
#         'Field Descriptions')

    dfRepo = pd.DataFrame(repo_sheet.get_all_records(head=3))
    xrefsBy4x4 = {}
    xrefsByTitle = {}

    for index,row in dfRepo[['Dataset Title','Socrata Link']].iterrows():
        xrefsByTitle[row['Dataset Title']] = row['Socrata Link']
        xrefsBy4x4[row['Socrata Link']] = row['Dataset Title']
        
    dfFields = pd.DataFrame(fields_sheet.get_all_records(head=1))
    fields = {}
    for index,row in dfFields.iterrows():
        s4x4 = row["Socrata ID"]
        of = row["Source Field Name"]
        tf = row["Full Field Name"]
        af = row["API Field Name"]
        if s4x4 in fields:
            fields[s4x4]["source"].append(of)
            fields[s4x4]["cim"].append(tf)
            fields[s4x4]["api"].append(af)
        else:
            fields[s4x4] = {}
            fields[s4x4]["source"] = []
            fields[s4x4]["cim"] = []
            fields[s4x4]["api"] = []
            
            fields[s4x4]["source"].append(of)
            fields[s4x4]["cim"].append(tf)
            fields[s4x4]["api"].append(af)
        
        
        
    return xrefsBy4x4,xrefsByTitle,fields



def getYrMoNamebyMonth(year,month):
    vals = []
    counts=0
    for error, count in sorted(statsYrMoName[year][month].items(), key=lambda item: item[1],reverse=True):
            val = []
            val.append(year)
            val.append(month)
            val.append(count)
            counts+=count
            val.append(error)
            vals.append(val)
    val=[]
    val.append("Total")
    val.append(month)
    val.append(counts)
    val.append("---")
    vals.append(val)
    return vals

def getErrors(name):
    vals = []
    print("ERROR ",name)
    for yr in sorted(statsNameYrMo[name]):
           #   for mo in range(1,13):
                  # if mo not in statsNameYrMo[name][yr]:
                  #       statsNameYrMo[name][yr][mo]=0
            for mo, moInfo in sorted(statsNameYrMo[name][yr].items(), key=lambda item: item[0],reverse=False):
                 val = []
                 val.append(yr)
                 val.append(mo)
                 val.append(moInfo['count'])
                 val.append(name)
                 vals.append(val)
    return vals

def getNameErrors(yr,mo,name):
    vals = []
    for line in statsNameYrMo[name][yr][mo]['errors']:
        vals.append(line)
    return '\n'.join(vals)

def toGoogleSheet(row):
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    # scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
    #          "https://www.googleapis.com/auth/drive.file",
    #               "https://www.googleapis.com/auth/drive"]

    path='../client_secret.json'
    gc=pygsheets.authorize(service_account_file=path)
    sh=gc.open('Changes/Fixes Requested by BIC')
    wk1=sh[0]
    wk1.append_table(row)

def markDone(row):
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    # scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
    #          "https://www.googleapis.com/auth/drive.file",
    #               "https://www.googleapis.com/auth/drive"]

    path='../client_secret.json'
    gc=pygsheets.authorize(service_account_file=path)
    sh=gc.open('ETL Errors Accounted For')
    wk1=sh[0]
    wk1.append_table(row)

def showError(row):
    layout = [
        [sg.Button("Quit")],
        [sg.Button("Write Log")],
        [sg.Button("Mark Done"),sg.Button("Mark Skip")],
        [sg.Multiline("",s=(50,10),key="-ERRORSINGLE-",font="CENTAUR 15 bold")],
        [sg.Text("Title: ",font="CENTAUR 15 bold"),
         sg.Multiline("",s=(50,2),key="-ERRORTITLE-",font="CENTAUR 15 bold",text_color="black")],
         [sg.Multiline("JIRA BIC-",s=(10,2),key="-ERRORJIRA-",font="CENTAUR 15 bold")],
        [sg.Text("NOTES: ",font="CENTAUR 15 bold"),
         sg.Multiline("",s=(20,10),key="-ERRORNOTES-",font="CENTAUR 15 bold")]
        
    ]
    window = sg.Window('Title', layout, finalize=True,metadata=row)
    window["-ERRORSINGLE-"].print(f"Date: {row[0]}", text_color='black')
    window["-ERRORSINGLE-"].print(f"Dataset: {row[1]}", text_color='black')
    window["-ERRORSINGLE-"].print(f"4x4: {row[2]}", text_color='black')
    window["-ERRORSINGLE-"].print(f"Title: {row[3]}", text_color='black')
    window["-ERRORSINGLE-"].print(f"Stitle: {row[5]}", text_color='black')
    
    window["-ERRORSINGLE-"].print(f"\nMessage:\n{row[4]}", text_color='red')
    if len(row[3]) > 10:
       window["-ERRORTITLE-"].print(f"{row[3]}", text_color='black')
    elif len(row[5]) > 10:
       window["-ERRORTITLE-"].print(f"{row[5]}", text_color='black')
        
        
                                  
                                  

def recentLogs(errorsByDate):
    xx = []
    colWidths = (10,10,10,80)
    rowColors = []
    count=0
    date0=list(errorsByDate.keys())[0]
    cols=("plum4")
    bg="white"
    bgs=[]
    count2=0
    for date in sorted(errorsByDate,reverse=True):
        string=f"Date: {date};;\n\n"
        for nn in range(len(errorsByDate[date]["name"])):
    #        print(f'    {errorsByDate[date]["name"][nn]}\n      M {errorsByDate[date]["msg"][nn]}\n      F{errorsByDate[date]["file"][nn]}\n      L {errorsByDate[date]["line"][nn]}\n')
        
          #  msg = textwrap.fill(errorsByDate[date]["msg"][nn],75)
            msg = errorsByDate[date]["msg"][nn]
            
            yy = [date,errorsByDate[date]["name"][nn],errorsByDate[date]["4x4"][nn],errorsByDate[date]["title"][nn],msg,errorsByDate[date]["stitle"][nn],errorsByDate[date]["uid"][nn]]
            xx.append(yy)
            if date != date0:
                count+=1
                if count%2 == 0:
                    cols = ("plum4")
                    bg="white"
                else:
                    cols = ("SteelBlue3")
                    bg="black"
            if count2%2 == 0:
                cols="plum4"
            else:
                cols="SteelBlue3"
            count2+=1
            date0=date
            rowColors.append(cols)
            bgs.append(bg)


    rowNums = [num for num in range(0,len(rowColors)+1)]
    colrw = list(zip(rowNums,rowColors))
    colrw=list(zip(rowNums,bgs,rowColors))
   

    header = ["Date","Dataset","4x4","Title","Message","Title Guess","UID"]

    layout = [
        [sg.Button("Quit")],
        [sg.Text("Errors for the Last 2 Months",font="CENTAUR 15 bold")],
        [sg.Table(values=xx,headings=header,visible_column_map=[True,True,True,True,True,False,False], size=(120, 30),row_height=40,row_colors=colrw,vertical_scroll_only=False,max_col_width=60,enable_events=True, key='-DAILY-',col_widths=colWidths)],
        [sg.Push(), sg.Button('Update')],
    ]
    window = sg.Window('Title', layout, finalize=True,metadata=xx)
    table = window["-DAILY-"]  
    table.bind('<Button-1>', "Click")
    window["-DAILY-"].Widget.column('#4', anchor='w') 
    return window

nameYrMoNames = sorted(statsNameYrMo.keys())
xrefsBy4x4,xrefsByTitle,fields = getXrefs()


## GUI

In [ ]:
months = ["","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
header_list = ["Year","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

right_click_menu = ['', ['Copy', 'Paste', 'Select All', 'Cut']]
###  PRIMARY LAYOUT
errorsByDate,errorsByName=getRecentErrors()
layout = [
         [sg.Text("BIC Update Errors")],
         [sg.Button('Close')],
         [sg.Button('Recent Errors')],
         [sg.Multiline("Update Info\nBe Certain to Connection via VPN before updating",s=(50,5),key="-UPDATES-")],
         [sg.Button("Update Logs This Month"),
           sg.Button("Update Logs Last Month"),
           sg.Button("Update Logs This Year")],

         
         [sg.Combo(nameYrMoNames,s=[65,10],expand_y=True,default_value=nameYrMoNames[0],key="-ERRORS-",enable_events=True)],
          [sg.Table(values=statsYrMoValues, 
               background_color='green',font="CENTAUR 15",
               auto_size_columns=False,enable_events=True,def_col_width=8,
               justification='center',alternating_row_color='brown',
               key='-TABLE-', headings = header_list)]
   ]
# Create the Window
sg.theme('Dark Green 5')
window = sg.Window('Output', layout,finalize=True,resizable=True)


#window2.move(window.current_location()[0]+600, window.current_location()[1])
table = window['-TABLE-']
table.bind('<Button-1>', "Click")
###  PRIMARY LOOP
while True:
    try:
     #   event, values = window.read()
        wid, event, values = sg.read_all_windows()
    #    window, event, values = sg.read_all_windows()
        print(event)
      #  print(values)
        if event == sg.WIN_CLOSED or event == 'Close':
            window.close()
            break
        elif event == "Quit":
            wid.close()
        elif event == '-ERRORS-':
            nameYrMoErrors = getErrors(values["-ERRORS-"])
            # if name == "Metadata Updater":
            #     print(nameYrMoErrors)

            layout3 = [    
                   [sg.Text(f"Summary of  {values['-ERRORS-']}")],
                   [sg.Table(values=nameYrMoErrors, max_col_width=30,
                   background_color='green',
                   auto_size_columns=True,enable_events=True,
                   justification='center',alternating_row_color='brown',
                   key='-TABLE3-', headings = ["Year","Month","Count","Error"])]
            ]
            window3 = sg.Window(f"Errors for {values['-ERRORS-']}", layout3,finalize=True,resizable=True)
        elif event == '-TABLE-':
            pass
        elif event == "Update Logs This Month":
            nfiles,files = updateLogs(1)
            string=f"{nfiles} Downloaded\n"
            for file in files:
                string+=f"{file}\n"
            window["-UPDATES-"].update(string)
            window["-TABLE-"].update(values=statsYrMoValues)
        elif event == "Update Logs Last Month":
            nfiles,files = updateLogs(2)
            string=f"{nfiles} Downloaded\n"
            for file in files:
                string+=f"{file}\n"
            window["-UPDATES-"].update(string)
            window["-TABLE-"].update(values=statsYrMoValues)

        elif event == "Update Logs This Year":
            nfiles,files = updateLogs(3)
            string=f"{nfiles} Downloaded\n"
            for file in files:
                string+=f"{file}\n"
            window["-UPDATES-"].update(string)
        elif event == "Mark Done" or event == "Mark Skip":
            toMark = [row[-1],]

        elif event == 'Recent Errors':

            w = recentLogs(errorsByDate)

        elif event == '-DAILY-Click':
            xx=wid.metadata
            
            table=wid["-DAILY-"]
            e = table.user_bind_event
            region = table.Widget.identify('region', e.x, e.y)
            if region == 'heading':
                row = 0
            elif region == 'cell':
                row = int(table.Widget.identify_row(e.y))
            elif region == 'separator':
                continue
            else:
                continue

            print(row,xx[row-1])

            string = f"Date: {xx[row-1][0]}\n\nDataset: {xx[row-1][1]}\n\nMessage: {xx[row-1][3]}\n"
            showError(xx[row-1])

        elif event == "Write Log":
            row=wid.metadata
            print(row)
            rowToWrite = [row[0],"Joe Comeaux",row[1],row[2],values["-ERRORTITLE-"],
                          row[4],values["-ERRORNOTES-"],values["-ERRORJIRA-"],row[-1]]
            count=0
            for val in rowToWrite:
                print(count,val)
                count+=1
#            yy = [date,errorsByDate[date]["name"][nn],errorsByDate[date]["4x4"][nn],errorsByDate[date]["title"][nn],msg,errorsByDate[date]["stitle"][nn]]
            toGoogleSheet(rowToWrite)

        elif event == '-TABLE-Click':
            table = window['-TABLE-']            
            e = table.user_bind_event
            region = table.Widget.identify('region', e.x, e.y)
            if region == 'heading':
                row = 0
            elif region == 'cell':
                row = int(table.Widget.identify_row(e.y))
            elif region == 'separator':
                continue
            else:
                continue
            column = int(table.Widget.identify_column(e.x)[1:])
            year = statsYrMoValues[row-1][0]
            month = column-1
            yrMoNameValues = getYrMoNamebyMonth(year,month)
            layout2 = [    
                   [sg.Text(f"Errors for {months[month]}, {year}")],
                   [sg.Button('Close')],
                   [sg.Table(values=yrMoNameValues, max_col_width=30,
                   background_color='green',
                   auto_size_columns=True,enable_events=True,
                   justification='center',alternating_row_color='brown',
                   key='-TABLE2-', headings = ["Year","Month","Count","Error"])]
            ]
            window2 = sg.Window(f"All Errors for {months[month]}, {year}", layout2,finalize=True,resizable=True)
            table2 = window2['-TABLE2-']
            table2.bind('<Button-1>', "Click")

            while True:

                event, values = window2.read()

            #    window, event, values = sg.read_all_windows()
                if event == sg.WIN_CLOSED or event == 'Close':
                    window2.close()
            #        sys.exit(1)
                    what = "QUIT"
                    break
                elif event == "Quit":
                    wid.close()
                elif event == '-TABLE2-':
                    pass
                elif event == '-TABLE2-Click':
                    e = table2.user_bind_event
                    region = table2.Widget.identify('region', e.x, e.y)
                    if region == 'heading':
                        row = 0
                    elif region == 'cell':
                        row = int(table2.Widget.identify_row(e.y))
                    elif region == 'separator':
                        continue
                    else:
                        continue
                    column = int(table2.Widget.identify_column(e.x)[1:])
                    #year = statsYrMoValues[row-1][0]
                    # month = column-1
                    error = yrMoNameValues[row-1][3]
                    nameErrorsbyYearMonth = getNameErrors(year,month,error)

                    layout4 = [    
                           [sg.Text(f"Errors for {months[month]}, {year}")],
                           [sg.Button('Close')],
                           [sg.Multiline(nameErrorsbyYearMonth,size=[80,20],horizontal_scroll=True,right_click_menu=right_click_menu)]

                    ]  
                    window4 = sg.Window(f"All Errors for {months[month]}, {year}", layout4,finalize=True,resizable=True)
                    while True:

                        event, values = window4.read()
                    #    window, event, values = sg.read_all_windows()
                        if event == sg.WIN_CLOSED or event == 'Close':
                            window4.close()
                    #        sys.exit(1)
                            what = "QUIT"
                            break
    except Exception as err:
        print("ERROR ",err)


## EXTRA

In [ ]:
errD,errN = getRecentErrors()

In [ ]:
ids

In [ ]:
errorsByDate

In [ ]:

def getRecentErrors():
    files = []
    print(path)
    for x in os.listdir(path):
            if re.findall("^log.json\.\d+",x):
                 # files2Process.append(x)
                 files.append(x)

    x = datetime.datetime.today() - datetime.timedelta(days=30)
    monthM = x.month  
    yearM = x.year
    ff=f"log_backup_{yearM}_{monthM}.json"
    files.append(ff)
    errorsByDate = {}
    errorsByName = {}
    print(files)
    bad = []
    for file in files:
        print(file)
        with open(f"{path}/{file}") as jsf:

            try:
                log=[]
                nline=0
                for line in jsf:
                    try: 
                         xl=line.lower()
                         
                         if "error" in xl and "metadata" not in xl:
                    #     if "error" in xl :
                             log.append(ast.literal_eval(line))
                    except:
                       bad.append(line)
                       print("BAD ",line)
                    nline+=1

                for line in log:
                  #  print(line['time'],line['name'])
                    tim = line['time']
                    
                    spl = tim.split("T")
                    tim = spl[0]
                    name = line['name'].strip()
                    msg= line['msg'].strip()

                    stitle=""
                    xm = msg.find("Full ETL failure for ")
                    if xm > -1:
                       stitle = msg[xm+20:].split(":")[0]
                       stitle=stitle.replace(" at load","")

                    xm = msg.find("Error Loading ")
                    if xm > -1:
                       stitle = msg[xm+14:]

                    xm = msg.find("Error Extracting ")
                    if xm > -1:
                       stitle=msg[xm+16:]

                    xm = msg.find("Error Transforming ")
                    if xm > -1:
                       stitle=msg[xm+18:]

                #   
                    titl=""
                    s4x4=""
                    w4x4=""
                    try:
                        s4x4s = re.findall("[\w]{3,4}-[\w]{3,4}",line['msg'])
                       

                        if len(s4x4s) > 0:
                          for s4x4 in s4x4s:
                             if s4x4 in xrefsBy4x4:
                                titl = xrefsBy4x4[s4x4]
                                w4x4=s4x4
                                

                    except:
                        titl=""


                    if tim in errorsByDate:
                        errorsByDate[tim]['name'].append(name)
                        errorsByDate[tim]['msg'].append(msg)
                        errorsByDate[tim]['line'].append(line)
                        errorsByDate[tim]['file'].append(file)
                        errorsByDate[tim]['title'].append(titl)
                        errorsByDate[tim]['stitle'].append(stitle)
                        errorsByDate[tim]['4x4'].append(w4x4)

                    else:
                        errorsByDate[tim] = {}
                        errorsByDate[tim]['name'] = []
                        errorsByDate[tim]['msg'] = []
                        errorsByDate[tim]['line'] = []
                        errorsByDate[tim]['file'] = []
                        errorsByDate[tim]['title'] = []
                        errorsByDate[tim]['stitle'] = []
                        errorsByDate[tim]['4x4'] = []



                        errorsByDate[tim]['name'].append(name)
                        errorsByDate[tim]['msg'].append(msg)
                        errorsByDate[tim]['line'].append(line)
                        errorsByDate[tim]['file'].append(file)
                        errorsByDate[tim]['title'].append(titl)
                        errorsByDate[tim]['stitle'].append(stitle)
                        errorsByDate[tim]['4x4'].append(w4x4)


                    if name in errorsByName:
                            errorsByName[name]['time'].append(tim)
                            errorsByName[name]['msg'].append(msg)
                            errorsByName[name]['line'].append(line)
                            errorsByName[name]['file'].append(file)
                            errorsByName[name]['title'].append(titl)
                    else:
                        errorsByName[name] = {}
                        errorsByName[name]['time'] = []
                        errorsByName[name]['msg'] = []
                        errorsByName[name]['line'] = []
                        errorsByName[name]['file'] = []
                        errorsByName[name]['title'] = []

                        errorsByName[name]['time'].append(tim)
                        errorsByName[name]['msg'].append(msg)
                        errorsByName[name]['line'].append(line)
                        errorsByName[name]['file'].append(file)
                        errorsByName[name]['title'].append(titl)

            except Exception as err:
                print(err)
                print(line)
    return errorsByDate,errorsByName
    
errorsByDate,errorsByName=getRecentErrors()

In [ ]:
for date in sorted(errorsByDate,reverse=True):
    print(f"Date: {date};;")
    for nn in range(len(errorsByDate[date]["name"])):
#        print(f'    {errorsByDate[date]["name"][nn]}\n      M {errorsByDate[date]["msg"][nn]}\n      F{errorsByDate[date]["file"][nn]}\n      L {errorsByDate[date]["line"][nn]}\n')
        print(f'    Dataset: {errorsByDate[date]["name"][nn]}\n') 
        print(f'    Message: {errorsByDate[date]["msg"][nn]}\n')   
        print(f'    Title: {errorsByDate[date]["title"][nn]}\n')
        print(f'    STitle: {errorsByDate[date]["stitle"][nn]}\n')                            
        print(f'    4x4: {errorsByDate[date]["4x4"][nn]}\n\n')                            

In [ ]:
import PySimpleGUI as sg
import textwrap

def toGoogleSheet(row):
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    # scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
    #          "https://www.googleapis.com/auth/drive.file",
    #               "https://www.googleapis.com/auth/drive"]

    path='../client_secret.json'
    gc=pygsheets.authorize(service_account_file=path)
    sh=gc.open('Changes/Fixes Requested by BIC')
    wk1=sh[0]
    wk1.append_table(row)


def showError(row):
    layout = [
        [sg.Button("Quit")],
        [sg.Button("Write Log")],
        [sg.Multiline("",s=(50,10),key="-ERRORSINGLE-",font="CENTAUR 15 bold")],
        [sg.Multiline("",s=(50,2),key="-ERRORTITLE-",font="CENTAUR 15 bold",text_color="black")],
         [sg.Multiline("JIRA BIC-",s=(10,2),key="-ERRORJIRA-",font="CENTAUR 15 bold")],
        [sg.Multiline("Notes",s=(20,10),key="-ERRORNOTES-",font="CENTAUR 15 bold")]
        
    ]
    window = sg.Window('Title', layout, finalize=True,metadata=row)
    window["-ERRORSINGLE-"].print(f"Date: {row[0]}", text_color='black')
    window["-ERRORSINGLE-"].print(f"Dataset: {row[1]}", text_color='black')
    window["-ERRORSINGLE-"].print(f"4x4: {row[2]}", text_color='black')
    window["-ERRORSINGLE-"].print(f"Title: {row[3]}", text_color='black')
    window["-ERRORSINGLE-"].print(f"Stitle: {row[5]}", text_color='black')
    
    window["-ERRORSINGLE-"].print(f"Message:\n {row[4]}", text_color='red')
    if len(row[3]) > 10:
       window["-ERRORTITLE-"].print(f"{row[3]}", text_color='black')
    elif len(row[5]) > 10:
       window["-ERRORTITLE-"].print(f"{row[5]}", text_color='black')
        
        
                                  
                                  

def recentLogs(errorsByDate):
    xx = []
    colWidths = (10,10,10,80)
    rowColors = []
    count=0
    date0=list(errorsByDate.keys())[0]
    cols=("plum4")
    bg="white"
    bgs=[]
    count2=0
    for date in sorted(errorsByDate,reverse=True):
        string=f"Date: {date};;\n\n"
        for nn in range(len(errorsByDate[date]["name"])):
    #        print(f'    {errorsByDate[date]["name"][nn]}\n      M {errorsByDate[date]["msg"][nn]}\n      F{errorsByDate[date]["file"][nn]}\n      L {errorsByDate[date]["line"][nn]}\n')
        
            msg = textwrap.fill(errorsByDate[date]["msg"][nn],75)
            yy = [date,errorsByDate[date]["name"][nn],errorsByDate[date]["4x4"][nn],errorsByDate[date]["title"][nn],msg,errorsByDate[date]["stitle"][nn]]
            xx.append(yy)
            if date != date0:
                count+=1
                if count%2 == 0:
                    cols = ("plum4")
                    bg="white"
                else:
                    cols = ("SteelBlue3")
                    bg="black"
            if count2%2 == 0:
                cols="plum4"
            else:
                cols="SteelBlue3"
            count2+=1
            date0=date
            rowColors.append(cols)
            bgs.append(bg)


    rowNums = [num for num in range(0,len(rowColors)+1)]
    colrw = list(zip(rowNums,rowColors))
    colrw=list(zip(rowNums,bgs,rowColors))
   

    header = ["Date","Dataset","4x4","Title","Message","Title Guess"]

    layout = [
        [sg.Table(values=xx,headings=header,visible_column_map=[True,True,True,True,True,False], size=(120, 30),row_height=40,row_colors=colrw,vertical_scroll_only=False,max_col_width=60,enable_events=True, key='-DAILY-',col_widths=colWidths)],
        [sg.Push(), sg.Button('Update')],
    ]
    window = sg.Window('Title', layout, finalize=True,metadata=xx)
    table = window["-DAILY-"]  
    table.bind('<Button-1>', "Click")
    window["-DAILY-"].Widget.column('#4', anchor='w') 
    return window
    
window = recentLogs(errorsByDate)
table=window["-DAILY-"]
xx=window.metadata
while True:
    try: 
        #event, values = window.read()
        wid, event, values = sg.read_all_windows()
        print(event)
        if event == sg.WIN_CLOSED:
            break
        if event == "Quit":
            wid.close()
            
        elif event == '-DAILY-Click':
            e = table.user_bind_event
            region = table.Widget.identify('region', e.x, e.y)
            if region == 'heading':
                row = 0
            elif region == 'cell':
                row = int(table.Widget.identify_row(e.y))
            elif region == 'separator':
                continue
            else:
                continue
                
            print(row,xx[row-1])
            
            string = f"Date: {xx[row-1][0]}\n\nDataset: {xx[row-1][1]}\n\nMessage: {xx[row-1][3]}\n"
            showError(xx[row-1])
            
        elif event == "Write Log":
            row=wid.metadata
            rowToWrite = [row[0],"Joe Comeaux",row[1],row[2],values["-ERRORTITLE-"],
                          row[4],values["-ERRORNOTES-"],values["-ERRORJIRA-"]]
            count=0
            for val in rowToWrite:
                print(count,val)
                count+=1
#            yy = [date,errorsByDate[date]["name"][nn],errorsByDate[date]["4x4"][nn],errorsByDate[date]["title"][nn],msg,errorsByDate[date]["stitle"][nn]]
            toGoogleSheet(rowToWrite)
    except Exception as err:
        print(err)

window.close()



In [ ]:
When	Who	Dataset(s)	4x4	Title	Message	Notes																			

In [ ]:
xx

In [ ]:
items = '\n'.join([
    'The earth or globe, considered as a planet.',
    'A particular division of the earth.',
    'The earth or a part of it, with its inhabitants, affairs, etc., during a particular period.',
    'Humankind; the human race; humanity.',
    'The public generally.',
    'The class of persons devoted to the affairs, interests, or pursuits of this life.',
])

new_items = '\n'.join([f'Line {i+1}' for i in range(10)])

layout = [
    [sg.Multiline(default_text=items, size=(30, 8), disabled=True,  enable_events=True, key='Multiline')],
    [sg.Push(), sg.Button('Update')],
]
window = sg.Window('Title', layout, finalize=True)
multiline = window['Multiline'].widget

# Disable default bindings
bindtags = list(multiline.bindtags())
bindtags.remove("Text")
multiline.bindtags(tuple(bindtags))

# Bind to click event
window['Multiline'].bind('<Button-1>', " Click")

# Bind to mouse wheel
def yscroll(event, widget):
    if event.num == 5 or event.delta < 0:
        widget.yview_scroll(1, "unit")
    elif event.num == 4 or event.delta > 0:
        widget.yview_scroll(-1, "unit")
multiline.bind('<MouseWheel>', lambda event, widget=multiline:yscroll(event, widget))

# Set line spacing between items
multiline.configure(spacing1=4, spacing2=0, spacing3=4)

while True:

    event, values = window.read()

    if event == sg.WIN_CLOSED:
        break
    elif event == 'Multiline Click':
        e = window['Multiline'].user_bind_event
        line, column = multiline.index(f"@{e.x},{e.y}").split(".")
        multiline.tag_remove('sel', "1.0", 'end')
        multiline.tag_add('sel', f'{line}.0', f'{line}.end')
        text = multiline.selection_get()
        print(text)
    elif event == 'Update':
        window['Multiline'].update(new_items)

window.close()

In [ ]:
import PySimpleGUI as sg
import textwrap

a = textwrap.fill(" Name: JC Type:Very Long Some very long sentiend with a lot of info and other thing Title, more long shit",10)

b = " Name: JC\nType:Very Long\nSome very long sentiend with a lot of info and other thing\nTitle, more long shit"
values = [" Name: JC\nType:Very Long\nSome very long sentiend with a lot of info and other thing\nTitle, more long shit",
          " Name: JC\nType:Very Long\nSome very long sentiend with a lot of info and other thing\nTitle, more long shit",
          " Name: JC\nType:Very Long\nSome very long sentiend with a lot of info and other thing\nTitle, more long shit"]



layout = [ [sg.Button("Quit")],
         [sg.Multiline(a,s=(12,10),key="TEXT1")],
         [sg.Multiline(b,"Errs",s=(12,2))]
          
       ]
#+sg.WRITE_ONLY_KEY
window = sg.Window('Output', layout,finalize=True,resizable=True)

window['TEXT1'].print(1,2,3,4,end='', text_color='red', background_color='yellow')
# multiline = window['TEXT1'].widget

# # Disable default bindings
# bindtags = list(multiline.bindtags())
# bindtags.remove("Text")
# multiline.bindtags(tuple(bindtags))

# # Bind to click event
# window['Multiline'].bind('<Button-1>', " Click")

while True:
       event,values = window.read()
       print(event)
        
       if event == "Quit":
            break
            
window.close()

print(values)

In [ ]:
def toGoogleSheet(row):
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('../client_secret.json',
     scope)
    client = gspread.authorize(creds)

    gc = gspread.service_account("../client_secret.json")
    for gg in gc.list_spreadsheet_files():
         print("GGGGG ",gg)
            
            
#   sheet = client.open('Changes/Fixes Requested by BIC').worksheet('Sheet1')
    sheet = client.open('Changes/Fixes Requested by BIC')
    sht = sheet.get_worksheet(0)
    
    sht.append_table(["Today","Joe C","cods business nonprofit","yesterday"])    

    dfRepo = pd.DataFrame(sheet.get_all_records(head=1))
    print(dfRepo.head())
            
getX()

In [ ]:
sheet.append_table(["Today","Joe C","cods business nonprofit","yesterday"])

In [ ]:
import pygsheets
path='../client_secret.json'
gc=pygsheets.authorize(service_account_file=path)
sh=gc.open('Changes/Fixes Requested by BIC')
wk1=sh[0]
wk1.append_table(["Today","Joe C","cods business nonprofit","yesterday"])